# `compound_lemma_processor` — Usage Notebook

Demonstrates the core algorithm behind `scripts/compound_lemma_processor.py`.

**What the script does**

Compound nouns in the corpus appear as groups of `N` / `N;@2` / `N;@3` … sibling
lines that share the same syntactic prefix up to a bare `N` marker column. The
script inserts a shared lemma ID at that marker column and creates/refines the
corresponding dictionary entry.

**Algorithm**
1. *(Optional)* **NP expansion**: detect NPs whose direct children are all `N`-at
   tags, insert a grouping bare `N` before them.
2. **Group detection** (`find_compound_groups`): find all consecutive sequences
   of lines with a shared marker-`N` column and component lemma IDs.
3. **Layered pairing** (`pair_components_layered`): for k ≥ 2 components, pair
   left-to-right iteratively: (A,B)→AB, (AB,C)→ABC, …
4. **ID insertion**: insert the outermost compound ID at `marker_col+1`.

All cells read from `data/` — no files are modified.

---
## 0. Setup

In [ ]:
import sys
import os
import re

sys.path.insert(0, os.path.join('..', 'src'))

DICT_FILE   = os.path.join('..', 'data', 'xml', 'dict', 'dictionary.xml')
CORPUS_FILE = os.path.join('..', 'data', 'xml', 'text', 'EN_01.xml')
TEXT_DIR    = os.path.join('..', 'data', 'xml', 'text')

print("Python:", sys.version.split()[0])


---
## 1. Corpus Line Structure for Compound Nouns

In the corpus a two-noun compound looks like:

```
IP-MAT,NP,N,L050001,N,L050002,LOG,ama
IP-MAT,NP,N,L050001,N;@2,L050003,LOG,tu
```

- The bare `N` at position 2 is the **marker column** — this is where the compound
  ID will be inserted.
- `N` at position 4 (component 1) and `N;@2` at position 4 (component 2) each carry
  their own lemma ID in the slot immediately after them.
- After processing, a compound ID `L050NNN` is inserted at position 3 (marker_col + 1):

```
IP-MAT,NP,N,L05NNN,L050001,N,L050002,LOG,ama
IP-MAT,NP,N,L05NNN,L050001,N;@2,L050003,LOG,tu
```

In [ ]:
from coj.core.corpus import CorpusDocument
from coj.core.tags import strip_disambig

LEMMA_PAT = re.compile(r'^[A-Za-z]\d+[a-z]*$')

doc = CorpusDocument.from_file(CORPUS_FILE)

# Find lines where a C-N or C-NP tag appears in the syntactic path
compound_candidates = []
for cl in doc.all_corpus_lines():
    if any(strip_disambig(t) in ('C-N', 'C-NP') for t in cl.fields):
        compound_candidates.append(cl)

print(f"Lines with C-N or C-NP tags in {os.path.basename(CORPUS_FILE)}: {len(compound_candidates)}")
print("\nFirst 5 examples:")
for cl in compound_candidates[:5]:
    print(f"  {cl.to_text()[:90]}")


---
## 2. Finding the Marker Column

The `_find_marker_col()` function locates the column index of the bare `N` that acts
as the compound group marker.  The contract:

- The marker `N` must NOT be immediately followed by a lemma ID (otherwise the compound
  is already annotated).
- The next field after the marker `N` must be another `N`-at tag (a component).
- That component N-at tag must be immediately followed by a lemma ID.

In [ ]:
from compound_lemma_processor import _is_bare_n, _is_annotated_n_leaf, _elem_children as _script_elem_children

# In the XML tree, a compound group is identified by:
# - A bare <N> element (no lemma attr) with 2+ annotated <N> leaf children
# Demonstrate these predicates on synthetic ET elements
import xml.etree.ElementTree as ET

def make_n_leaf(lemma=None, form="test"):
    e = ET.Element("N")
    e.set("form", form)
    if lemma:
        e.set("lemma", lemma)
    return e

def make_bare_n(*children):
    e = ET.Element("N")
    for c in children:
        e.append(c)
    return e

# Test cases
leaf_annotated  = make_n_leaf(lemma="L000100", form="ama")
leaf_unannotated = make_n_leaf(form="tu")
bare_with_children = make_bare_n(
    make_n_leaf(lemma="L000100", form="ama"),
    make_n_leaf(lemma="L000200", form="tu"),
)
bare_annotated = make_bare_n(make_n_leaf(lemma="L000100", form="ama"))
bare_annotated.set("lemma", "L050001")

print("XML predicate demonstration:")
print(f"  _is_annotated_n_leaf(leaf with lemma)   : {_is_annotated_n_leaf(leaf_annotated)}")
print(f"  _is_annotated_n_leaf(leaf no lemma)     : {_is_annotated_n_leaf(leaf_unannotated)}")
print(f"  _is_bare_n(bare <N> with children)      : {_is_bare_n(bare_with_children)}")
print(f"  _is_bare_n(bare <N> with lemma attr)    : {_is_bare_n(bare_annotated)}")
print()
print("A compound group is a bare <N> whose ALL element children are annotated <N> leaves:")
children = _script_elem_children(bare_with_children)
print(f"  children count: {len(children)}")
print(f"  all annotated N leaves: {all(_is_annotated_n_leaf(c) for c in children)}")


---
## 3. Group Detection

Import and use the script's `find_compound_groups()` to scan a real corpus file.

We replicate the group-detection logic here to show what it finds.

In [ ]:

# Use the XML-native group detection from the script
sys.path.insert(0, os.path.join('..', 'scripts', 'processors'))
from compound_lemma_processor import find_compound_groups_xml

doc_elem = doc._get_doc_elem()
groups = find_compound_groups_xml(doc_elem)

print(f"Compound groups found in {os.path.basename(CORPUS_FILE)}: {len(groups)}")
print("\nSample groups:")
for g in groups[:5]:
    components = g['components']
    forms = [c['text_form'] for c in components]
    ids   = [c['comp_id']   for c in components]
    print(f"  {len(components)} components: {list(zip(ids, forms))}")


---
## 4. Layered Binary Pairing

For a group of k components, `pair_components_layered()` builds intermediate compound
IDs left-to-right:

- **k=2**: `(A, B) → AB_id`
- **k=3**: `(A, B) → AB_id`, then `(AB_id, C) → ABC_id`
- **k=4**: `(A, B) → AB_id`, `(C, D) → CD_id`, then `(AB_id, CD_id) → ABCD_id`

The final ID in the chain is inserted at `marker_col+1` in each line of the group.

In [5]:
def pair_components_layered_demo(components: list[dict]) -> list[tuple]:
    """Simulate layered pairing; return list of (combined_form, layer)."""
    slots = [(c['comp_id'], c['text_form']) for c in components]
    results = []
    layer = 0
    new_id_counter = [50001]

    while len(slots) > 1:
        new_slots = []
        k = 0
        while k < len(slots):
            if k + 1 < len(slots):
                id1, form1 = slots[k]
                id2, form2 = slots[k + 1]
                new_id = f"L{new_id_counter[0]:06d}"
                new_id_counter[0] += 1
                combined_form = form1 + form2
                new_slots.append((new_id, combined_form))
                results.append((new_id, combined_form, layer))
                k += 2
            else:
                new_slots.append(slots[k])
                k += 1
        slots = new_slots
        layer += 1

    return results

# Test cases
examples = [
    [{'comp_id': 'L000100', 'text_form': 'ama'},
     {'comp_id': 'L000200', 'text_form': 'tu'}],
    [{'comp_id': 'L000100', 'text_form': 'taka'},
     {'comp_id': 'L000200', 'text_form': 'ma'},
     {'comp_id': 'L000300', 'text_form': 'para'}],
    [{'comp_id': 'L000100', 'text_form': 'a'},
     {'comp_id': 'L000200', 'text_form': 'si'},
     {'comp_id': 'L000300', 'text_form': 'pi'},
     {'comp_id': 'L000400', 'text_form': 'ki'}],
]

for comps in examples:
    results = pair_components_layered_demo(comps)
    input_forms = [c['text_form'] for c in comps]
    print(f"  Input: {input_forms}")
    for cid, form, layer in results:
        print(f"    layer {layer}: {cid} = '{form}'")
    outermost = results[-1]
    print(f"    → insert {outermost[0]} at marker column")
    print()

  Input: ['ama', 'tu']
    layer 0: L050001 = 'amatu'
    → insert L050001 at marker column

  Input: ['taka', 'ma', 'para']
    layer 0: L050001 = 'takama'
    layer 1: L050002 = 'takamapara'
    → insert L050002 at marker column

  Input: ['a', 'si', 'pi', 'ki']
    layer 0: L050001 = 'asi'
    layer 0: L050002 = 'piki'
    layer 1: L050003 = 'asipiki'
    → insert L050003 at marker column



---
## 5. NP Expansion (Optional)

When `NP_EXPANSION = True`, the script first looks for NPs whose **all** immediate
direct children are `N`-at tags without an existing marker `N` above them.

For example:
```
...,NP,N,L000100,LOG,ama      ← N is a direct child of NP
...,NP,N;@2,L000200,LOG,tu   ← N;@2 is sibling
```
After NP expansion, a grouping bare `N` is inserted at the NP child level:
```
...,NP,N,N,L000100,LOG,ama
...,NP,N,N;@2,L000200,LOG,tu
```
This makes the group detectable by `find_compound_groups()`.

In [ ]:
from compound_lemma_processor import _is_annotated_n_leaf

# Count NP candidates for expansion in the XML tree
# An NP whose ALL direct element children are annotated <N> leaves

def _elem_children(elem):
    return [c for c in elem if c.tag != "comment"]

np_candidates = []

def _find_np_candidates(elem):
    children = _elem_children(elem)
    if elem.tag == "NP":
        n_children = [c for c in children if c.tag == "N"]
        if (len(n_children) >= 2
                and n_children == children
                and all(_is_annotated_n_leaf(c) for c in n_children)):
            np_candidates.append((elem, n_children))
            return
    for child in children:
        _find_np_candidates(child)

for block in doc_elem:
    if block.tag == "block":
        for child in _elem_children(block):
            _find_np_candidates(child)

print(f"NP candidates for expansion in {os.path.basename(CORPUS_FILE)}: {len(np_candidates)}")
if np_candidates:
    elem, children = np_candidates[0]
    print(f"\nExample: <NP> with {len(children)} annotated <N> leaf children")
    print("  Children:", [(c.tag, c.get('lemma'), c.get('form')) for c in children])
    print("  → After expansion: a bare <N> wrapper inserted around these children")


---
## 6. Dictionary Interaction

For each compound pair, the script calls `resolve_compound()` which:
1. Checks if an entry whose `.FORM` matches the concatenated form already exists
   (`DICT_SEARCH = True`).
2. If found and missing `.COMPOUND` back-references, adds them (`DICT_REFINE = True`).
3. If not found, creates a new entry (`DICT_ENTRY_CREATE = True`).

The `.COMPOUND ref_target=<ID>  <form>` field uses the **dictionary's canonical
`.FORM`** (not the corpus text form) to ensure stable references.

Below we show how to check whether a compound form already exists in the dictionary.

In [ ]:
from coj.core.dictionary import Dictionary

d = Dictionary.from_file(DICT_FILE)

def find_compound_in_dict(form, dictionary):
    """Return the entry ID if the combined form exists in the dictionary."""
    for entry in dictionary:
        if form in entry.get_all('.FORM'):
            return entry.eid
    return None

# Test with compound forms from detected groups
print("Compound form lookups:")
test_forms = []
for g in groups[:10]:
    combined = ''.join(c['text_form'] for c in g['components'])
    test_forms.append(combined)

for form in set(test_forms):
    eid = find_compound_in_dict(form, d)
    status = f"found: {eid}" if eid else "not in dictionary → would create new entry"
    print(f"  {form:20s}  {status}")

if not test_forms:
    print("  (no compound groups detected in this file)")


In [8]:
from coj.core.kana import phonemic_to_kana

def build_compound_entry_preview(
    new_id: str,
    comp1_id: str, comp1_dict_form: str,
    comp2_id: str, comp2_dict_form: str,
    combined_text_form: str,
) -> str:
    """Show what a new compound dictionary entry would look like."""
    combined_kana = phonemic_to_kana(combined_text_form)
    lines = [
        f"=== {new_id}",
        f".GLOSS\t{combined_text_form}",
        ".MEANING\t[compound noun]",
        f".FORM\t{combined_text_form}",
        f".KANA\t{combined_kana}",
        ".POS\tnoun",
        f".COMPOUND\tref_target={comp1_id}  {comp1_dict_form}",
        f".COMPOUND\tref_target={comp2_id}  {comp2_dict_form}",
    ]
    return '\n'.join(lines)

# Use first detected group
if groups:
    g = groups[0]
    c0, c1 = g['components'][0], g['components'][1]
    # Look up canonical .FORM from dictionary
    e0 = d.get(c0['comp_id'])
    e1 = d.get(c1['comp_id'])
    dict_form0 = e0.get_first('.FORM') if e0 else c0['text_form']
    dict_form1 = e1.get_first('.FORM') if e1 else c1['text_form']
    combined   = c0['text_form'] + c1['text_form']

    preview = build_compound_entry_preview(
        'L050001', c0['comp_id'], dict_form0, c1['comp_id'], dict_form1, combined
    )
    print("Simulated new compound dictionary entry:")
    print(preview)

---
## 7. Inserting the Compound ID into Corpus Lines

Once the outermost compound ID is resolved, it is inserted at `marker_col + 1` in
every line of the group.

In [ ]:
if groups:
    g = groups[0]
    bare_n = g['bare_n_elem']
    components = g['components']
    compound_id = 'L050001'  # simulated

    print("Before insertion (component lines):")
    for c in components:
        print(f"  comp_id={c['comp_id']}  form={c['text_form']!r}")

    print(f"\nAfter insertion: bare <N> element gets lemma='{compound_id}'")
    print(f"  bare_n.set('lemma', '{compound_id}')")
    print("  (in production: bare_n.set('lemma', outermost_compound_id))")
else:
    print("No compound groups detected in", os.path.basename(CORPUS_FILE))


---
## 8. Corpus-Wide Statistics

How many compound groups would be detected across all corpus files?

In [ ]:
total_groups = 0
print(f"{'File':22s}  Groups")
print('-' * 32)
for fname in sorted(os.listdir(TEXT_DIR)):
    if not fname.endswith('.xml'):
        continue
    doc_i = CorpusDocument.from_file(os.path.join(TEXT_DIR, fname))
    cnt = len(find_compound_groups_xml(doc_i._get_doc_elem()))
    total_groups += cnt
    print(f"  {fname:20s}  {cnt:4d}")

print(f"\n  Total compound groups: {total_groups}")


---
## 9. Running the Script

Configure the `USER SETTINGS` block in `scripts/compound_lemma_processor.py`, then:

```bash
python3 scripts/compound_lemma_processor.py
```

Key settings:

| Setting | Default | Effect |
|---|---|---|
| `NP_EXPANSION` | `True` | Detect and expand NP-grouped N-at siblings before compound detection |
| `DICT_SEARCH` | `True` | Look up compound form in existing dictionary entries |
| `DICT_REFINE` | `True` | Add missing `.COMPOUND` lines to an existing entry |
| `DICT_ENTRY_CREATE` | `True` | Create a new entry when the compound is absent from the dictionary |
| `OVERWRITE_SOURCE` | `False` | Write `*_processed.txt` to `OUTPUT_FOLDER` instead of editing source |
| `LEMMA_START` | `50001` | Minimum numeric value for new compound IDs |
| `LEMMA_PREFIX` | `"L"` | Prefix for newly generated compound IDs |

The script iterates compound detection up to 20 times per file to handle deeply
nested compounds (e.g. a 4-component noun whose intermediate 2-component results
are themselves eligible for further grouping).